<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/5_transformer_90_model/5_3_transformer_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5_3_model_transformers

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Instalación de librerías


### 0.2. Importación de librerías


In [1]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [2]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
Platform: Linux-6.6.105+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.3
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


In [4]:
import psutil

def ram_usage():
    ram = psutil.virtual_memory()
    used = ram.used / (1024**3)
    free = ram.available / (1024**3)
    total = ram.total / (1024**3)

    print(f"RAM total:      {total:.2f} GB")
    print(f"RAM usada:      {used:.2f} GB")
    print(f"RAM disponible: {free:.2f} GB")

## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_data(data: str):

    data_path = f'{drive_path}/5_transformer_90_model/5_0_k_folds/fold_{fold}/{data}_{fold}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [6]:
k_folds = [1, 2, 3, 4, 5]

In [7]:
#mnq_train = {}
#mnq_valid = {}
#mnq_test  = {}

#for k in k_folds:
#    print(f'Cargando datos de Fold {k}..')
#    mnq_train[k] = load_data(str(k), 'train')
#    mnq_valid[k] = load_data(str(k), 'valid')
#    mnq_test[k]  = load_data(str(k), 'test')

### 1.2. Información de datasets


In [8]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [9]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [10]:
import json
# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
features_to_90 = features_dict["features_to_90"]

In [11]:
print(f'Listado de features para 90min: {features_to_90}')

Listado de features para 90min: ['ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']


## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### 2.0.1. Función para cargar ventanas

In [12]:
def load_windows_and_scaler(k: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz'
    path_valid  = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz'
    path_test   = f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/5_transformer_90_model/5_2_k_scaler/global_scaler.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    print(f'\tX_train_sc_{k} e y_train_{k} extraídos correctamente')
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    print(f'\tX_valid_sc_{k} e y_valid_{k} extraídos correctamente')
    X_test,  y_test  = data_test["X"],  data_test["y"]
    print(f'\tX_test_sc_{k} e y_test_{k} extraídos correctamente')

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### 2.0.2. Función para revisar información de ventanas

In [13]:
def xy_info(k, X_train, y_train, X_valid, y_valid, X_test, y_test, silent=False):
    import numpy as np
    import psutil

    if not silent:
        print(f"Información de {k}:")
        print("----------------------------------------")

    # Memoria total
    total_ram_gb = psutil.virtual_memory().total / (1024 ** 3)

    def print_set_info(nombre, X, y):
        if silent:
            return  # No imprimir nada

        n_samples = X.shape[0]
        size_X_gb = X.nbytes / (1024 ** 3)
        size_y_gb = y.nbytes / (1024 ** 3)
        total_gb = size_X_gb + size_y_gb
        perc_ram = (total_gb / total_ram_gb) * 100
        y_flat = np.ravel(y)

        print(f"Set de {nombre}:")
        print(f"\t{n_samples} ventanas")
        print(f"\tTamaño X: {size_X_gb:.3f} GB")
        print(f"\tTamaño y: {size_y_gb:.6f} GB")
        print(f"\tTOTAL: {total_gb:.3f} GB → {perc_ram:.1f}% RAM\n")

    # Mostrar info solo si silent=False
    print_set_info("entrenamiento", X_train, y_train)
    print_set_info("validación",    X_valid, y_valid)
    print_set_info("testeo",        X_test,  y_test)

    # Pesos = cantidad de ventanas
    w_train = X_train.shape[0]
    w_valid = X_valid.shape[0]
    w_test  = X_test.shape[0]

    return w_train, w_valid, w_test


### 2.1 Carga de ventanas

In [14]:
#Para verificar el formato de lo guardado.
#for k in k_folds:
#    print(f'Fold {k}:')
#    print('\tTrain:\t', np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_train_sc_{k}.npz').files)
#    print('\tValid:\t', np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_valid_sc_{k}.npz').files)
#    print('\tTest:\t',np.load(f'{drive_path}/5_transformer_90_model/5_2_k_scaler/fold_{k}/X_test_sc_{k}.npz').files)

In [15]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      1.35 GB
RAM disponible: 11.01 GB


In [16]:
# Diccionarios para almacenar datos escalados por fold
X_train_sc = {}
y_train_sc = {}
X_valid_sc = {}
y_valid_sc = {}
X_test_sc  = {}
y_test_sc  = {}
scalers    = {}

for k in k_folds:
    print(f'Fold {k}:')

    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler = load_windows_and_scaler(k)

    # Guardar todo en diccionarios
    X_train_sc[k] = X_train
    y_train_sc[k] = y_train

    X_valid_sc[k] = X_valid
    y_valid_sc[k] = y_valid

    X_test_sc[k]  = X_test
    y_test_sc[k]  = y_test

    scalers[k] = scaler

    print(f"  - Datos escalados cargados y almacenados en diccionarios.")
    print("-" * 40)

Fold 1:
	X_train_sc_1 e y_train_1 extraídos correctamente
	X_valid_sc_1 e y_valid_1 extraídos correctamente
	X_test_sc_1 e y_test_1 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 2:
	X_train_sc_2 e y_train_2 extraídos correctamente
	X_valid_sc_2 e y_valid_2 extraídos correctamente
	X_test_sc_2 e y_test_2 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 3:
	X_train_sc_3 e y_train_3 extraídos correctamente
	X_valid_sc_3 e y_valid_3 extraídos correctamente
	X_test_sc_3 e y_test_3 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
----------------------------------------
Fold 4:
	X_train_sc_4 e y_train_4 extraídos correctamente
	X_valid_sc_4 e y_valid_4 extraídos correctamente
	X_test_sc_4 e y_test_4 extraídos correctamente
  - Datos escalados cargados y almacenados en diccionarios.
-------------

In [17]:
#Como accedeR:
#Xtr = X_train_sc[3]   # X_train del fold 3
#ytr = y_train_sc[3]

In [18]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      5.96 GB
RAM disponible: 6.40 GB


In [19]:
pesos_folds = {}

for k in k_folds:
    w_train, w_valid, w_test = xy_info(
        k,
        X_train_sc[k],
        y_train_sc[k],
        X_valid_sc[k],
        y_valid_sc[k],
        X_test_sc[k],
        y_test_sc[k],
        silent=True   # evita imprimir
    )

    pesos_folds[k] = {
        "w_train": w_train,
        "w_valid": w_valid,
        "w_test":  w_test,
    }


In [20]:
pesos_folds

{1: {'w_train': 124279, 'w_valid': 24898, 'w_test': 27852},
 2: {'w_train': 149177, 'w_valid': 24898, 'w_test': 27852},
 3: {'w_train': 174075, 'w_valid': 24898, 'w_test': 27852},
 4: {'w_train': 198973, 'w_valid': 24898, 'w_test': 27852},
 5: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852}}

In [21]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      5.96 GB
RAM disponible: 6.40 GB


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [116]:
def load_metrics(subcarpeta: str, data: str):
    data_path = f'{drive_path}/5_model_90_transformer/5_3_model_transformer/{subcarpeta}/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [117]:
def metrics_verify(subcarpeta: str, data: str) -> bool:
    data_path = f'{drive_path}/5_model_90_transformer/5_3_model_transformer/{subcarpeta}/{data}.parquet'
    return os.path.exists(data_path)


In [125]:
def load_or_create_metrics (subcarpeta: str, data:str):
  if metrics_verify(subcarpeta, data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(subcarpeta, data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[2:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [126]:
transformers_metrics, metrics = load_or_create_metrics("5_3_0_trained_k_models", "0_transformers_metrics")

Las métricas no existen. Se crea el dataset transformers_metrics para almacenar las métricas


In [127]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


### 3.2. Función para guardar métricas

In [128]:
def save_metrics (metrics,  subcarpeta: str, metrics_name: str):
  metrics_path = f"{drive_path}/5_transformer_90_model/5_3_model_transformer/{subcarpeta}/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [88]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [89]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

# Entrenamiento de Transformers

## 4. Re-formateo más Encoder mínimo

Helper para re-formatear nuestras ventanas 2D a 3D que el formato que el modelo necesita.

### 4.1. Helper: de 2D (aplanado) a 3D (B, T, F)

Lo usamos para cada set y horizonte. Nuestro `window_size = 90` y los `n_features` depende del horizonte de tiempo. El siguiente código valida que `windows_size * n_features == X.shape[1]`

In [30]:
import numpy as np
import torch
import torch.nn as nn
import math

def reshape_windows(X_flat: np.ndarray, window_size: int, n_features: int) -> np.ndarray:
    """
    Convierte X de (N, window_size * n_features) a (N, window_size, n_features).
    Valida la consistencia del producto.
    """
    assert X_flat.ndim == 2, "Se esperaba X_flat con 2D (N, T*F)."
    N, TF = X_flat.shape
    assert window_size * n_features == TF, (
        f"Inconsistencia: {window_size} * {n_features} != {TF}"
    )
    return X_flat.reshape(N, window_size, n_features)

In [31]:
window_size = 90
features_base = ['open','high','close','low','volume']
features_90 = features_base + features_to_90
n_features_90 = len (features_90)
print(features_90)
print(n_features_90)


['open', 'high', 'close', 'low', 'volume', 'ire_60', 'rev_mom_z_90', 'roc_60', 'bb_60', 'momentum_5', 'roc_20', 'rev_mom_vol_z_60']
12


In [32]:
import gc

Xtr = {}
Xva = {}
Xte = {}

for k in k_folds:
    Xtr[k] = reshape_windows(X_train_sc[k], window_size, n_features_90)
    Xva[k] = reshape_windows(X_valid_sc[k], window_size, n_features_90)
    Xte[k] = reshape_windows(X_test_sc[k],  window_size, n_features_90)

    # Liberar las matrices 2D de este fold
    del X_train_sc[k], X_valid_sc[k], X_test_sc[k]
    gc.collect()

    print(f'Fold {k} re-shape completo y 2D liberado')

Fold 1 re-shape completo y 2D liberado
Fold 2 re-shape completo y 2D liberado
Fold 3 re-shape completo y 2D liberado
Fold 4 re-shape completo y 2D liberado
Fold 5 re-shape completo y 2D liberado


In [33]:
def mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte):
    for k in k_folds:
        print(f"\nShapes del Fold {k}")
        print(f"{'Set':<10}{'Shape (3D)':<25}")
        print("-" * 40)

        filas = [
            ("Train", Xtr[k].shape),
            ("Valid", Xva[k].shape),
            ("Test",  Xte[k].shape),
        ]

        for nombre, shape_3d in filas:
            print(f"{nombre:<10}{str(shape_3d):<25}")

In [34]:
mostrar_shapes_folds_simple(k_folds, Xtr, Xva, Xte)


Shapes del Fold 1
Set       Shape (3D)               
----------------------------------------
Train     (124279, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 2
Set       Shape (3D)               
----------------------------------------
Train     (149177, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 3
Set       Shape (3D)               
----------------------------------------
Train     (174075, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 4
Set       Shape (3D)               
----------------------------------------
Train     (198973, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852, 90, 12)          

Shapes del Fold 5
Set       Shape (3D)               
----------------------------------------
Train     (223871, 90, 12)         
Valid     (24898, 90, 12)          
Test      (27852

### 4.2. Encoder: (backbone + posición + TransformerEncoder)

Mi TimeSeriesEncoder

- Entrada: x con shape (B, T, F)
  - B = batch size
  - T = ventana temporal (p.ej. 90 pasos)
  - F = cantidad de features por minuto

- Salida: z con shape (B, T, D)
  - D = d_model (en tu caso 128)

Es decir: para cada paso temporal devuelve un embedding de dimensión 128

In [35]:
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 2048):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer("pe", pe.unsqueeze(0))  # (1, max_len, d_model)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, D)
        T = x.size(1)
        return x + self.pe[:, :T]

class TimeSeriesEncoder(nn.Module):
    """
    Proyección a d_model + PositionalEncoding + TransformerEncoder (sin cabeza).
    Devuelve embeddings por paso temporal: (B, T, d_model)
    """
    def __init__(
        self,
        input_dim: int,
        d_model: int = 128,
        #Hiperparámetros del modelo
        nhead: int = 8,
        num_layers: int = 2,
        dim_feedforward: int = 256,
        dropout: float = 0.1,
        activation: str = "gelu",
    ):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, d_model)
        self.pos_encoder = SinusoidalPositionalEncoding(d_model)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout,
            activation=activation,
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.dropout = nn.Dropout(dropout)

        # init
        nn.init.xavier_uniform_(self.input_proj.weight)
        nn.init.zeros_(self.input_proj.bias)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, T, F)
        z = self.input_proj(x)    # (B, T, D)
        z = self.pos_encoder(z)   # (B, T, D)
        z = self.encoder(z)       # (B, T, D)
        z = self.dropout(z)       # (B, T, D)
        return z

Creo un encoder por cada fold, con la misma arquitectura para todos los folds y pesos distintos (cada encoder_k es un modelo nuevo)

In [36]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Diccionario de encoders por fold
encoders = {}

for k in k_folds:
    encoders[k] = TimeSeriesEncoder(
        input_dim=n_features_90,
        # HIPERPARÁMETROS FIJADOS (Y PRÓXIMOS A TUNEAR)
        d_model=128,
        nhead=8,
        num_layers=2,
        # dim_feedforward=256,  # default
        # dropout=0.1,          # default
        # activation="gelu",    # default
    ).to(device)
    print(f"Encoder creado para fold {k}")

Encoder creado para fold 1
Encoder creado para fold 2
Encoder creado para fold 3
Encoder creado para fold 4
Encoder creado para fold 5


El siguiente código es un testeo rápido para verificar que:
  - Las ventanas del fold están correctamente cargadas.
  - En encoder funciona bien.
  - Las dimensiones de salida son las esperadas.

No está entrenando nada, solo está probando.

In [37]:
def verificar_encoder(fold, Xtr, Xva, Xte, encoders):
    print(f"\n=== Fold {fold} ===")

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # ---------- 1) Cargar ventanas 3D ----------
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    # ---------- 2) Mini-batches para inspección ----------
    xb_tr = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:64], dtype=torch.float32).to(device)

    # ---------- 3) Encoder del fold ----------
    encoder = encoders[fold]

    # ---------- 4) Pares para inspección ----------
    pairs = [
        (xb_tr, encoder, f"Fold{fold}-train"),
        (xb_va, encoder, f"Fold{fold}-valid"),
        (xb_te, encoder, f"Fold{fold}-test"),
    ]

    # ---------- 5) Ejecutar encoder ----------
    for xb, enc, tag in pairs:
        with torch.no_grad():
            z = enc(xb)
        print(tag, "→", z.shape)



In [38]:
for k in k_folds:
  verificar_encoder(k, Xtr, Xva, Xte, encoders)


=== Fold 1 ===
Fold1-train → torch.Size([64, 90, 128])
Fold1-valid → torch.Size([64, 90, 128])
Fold1-test → torch.Size([64, 90, 128])

=== Fold 2 ===
Fold2-train → torch.Size([64, 90, 128])
Fold2-valid → torch.Size([64, 90, 128])
Fold2-test → torch.Size([64, 90, 128])

=== Fold 3 ===
Fold3-train → torch.Size([64, 90, 128])
Fold3-valid → torch.Size([64, 90, 128])
Fold3-test → torch.Size([64, 90, 128])

=== Fold 4 ===
Fold4-train → torch.Size([64, 90, 128])
Fold4-valid → torch.Size([64, 90, 128])
Fold4-test → torch.Size([64, 90, 128])

=== Fold 5 ===
Fold5-train → torch.Size([64, 90, 128])
Fold5-valid → torch.Size([64, 90, 128])
Fold5-test → torch.Size([64, 90, 128])


Los resultados significan que:
- batch size = 64
- T = 90 pasos temporales
- d_model = 128 (dimensión del embedding por paso)

In [39]:
ram_usage()

RAM total:      12.67 GB
RAM usada:      6.36 GB
RAM disponible: 5.99 GB


## 5. Pooling (Sin cambiar enconder)

Tenemos dos opciones simples (no requieren modificar el encoder):

- `mean`: promedio temporal.
- `last`: último paso temporal.

In [40]:
import torch
import torch.nn as nn

class TemporalPooling(nn.Module):
    def __init__(self, mode: str = "mean"):
        super().__init__()
        assert mode in ("mean", "last"), "Soportado: 'mean' o 'last'"
        self.mode = mode

    def forward(self, z: torch.Tensor) -> torch.Tensor:
        # z: (B, T, D)
        if self.mode == "mean":
            return z.mean(dim=1)      # (B, D)
        else:  # "last"
            return z[:, -1, :]        # (B, D)

In [41]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:
    print(f"\n=== Fold {fold} ===")

    # 1) Cargar ventanas 3D del fold desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    # 2) Mini-batches
    xb_tr = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:64], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold desde diccionario
    enc = encoders[fold]

    # 4) Lista de sets de este fold
    pairs = [
        (xb_tr, enc, f"Fold{fold}-train"),
        (xb_va, enc, f"Fold{fold}-valid"),
        (xb_te, enc, f"Fold{fold}-test"),
    ]

    # 5) Pasar por encoder + pooling
    for xb, encoder, tag in pairs:
        with torch.no_grad():
            z  = encoder(xb)    # (64, T, d_model)
            p1 = pool_mean(z)   # (64, d_model)
            p2 = pool_last(z)   # (64, d_model)
        print(f'Para {tag}\n\tPool mean:\t{p1.shape}\tPool last:\t{p2.shape}')



=== Fold 1 ===
Para Fold1-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold1-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 2 ===
Para Fold2-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold2-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 3 ===
Para Fold3-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold3-test
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])

=== Fold 4 ===
Para Fold4-train
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-valid
	Pool mean:	torch.Size([64, 128])	Pool last:	torch.Size([64, 128])
Para Fold4-test

Intepretación:

  - Estamos tomando 64 ventanas de cada set (train/valid/test) del fold k, por lo tanto el batch es de tamaño 64.
  - El encoder devuelve secuencias (64, T, 128) y luego:
    - pool_mean(z) → comprime en (64, 128) (promedio temporal).
    - pool_last(z) → comprime en (64, 128) (último paso temporal).
  - Para todos los folds, la dimensión del embedding es 128, como se definió con d_model=128.

Que las shapes sean iguales entre folds es normal: todos usan el mismo d_model y el mismo batch_size.

Hemos verificado que:
  - Que Xtr_k, Xva_k, Xte_k tienen la forma correcta (pueden entrar al encoder).
  - Que los encoder_k están bien definidos y funcionan para todos los folds.
  - Que el TemporalPooling funciona y genera embeddings 2D (batch, 128) listos para una cabeza final (regresión/clasificación).

### 5.0. Verificación de valores entre folds

Veamos el contenido de las primeras filas para compararlas fold a fold, deberían ser distintos (otra distribución temporal, otros días, etc.), aunque la forma se la misma.

El siguiente código es para ver el contenido real (los valores numéricos) del embedding de cada fold, no solo las dimensiones.

#### 5.0.1. Primeros valores del embedding por fold

In [42]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:

    print(f"\n=== Fold {fold} ===")

    # 1) Cargar ventanas 3D desde diccionario
    Xtr_k = Xtr[fold]

    # 2) Mini-batch
    xb = torch.tensor(Xtr_k[:64], dtype=torch.float32).to(device)

    # 3) Encoder del fold desde diccionario
    enc = encoders[fold]

    # 4) Ejecutar encoder + pooling
    with torch.no_grad():
        z = enc(xb)               # (64, T, 128)
        p_mean = pool_mean(z)     # (64, 128)

    # 5) Mostrar valores numéricos reales del embedding
    print("Primeros 10 valores del embedding del fold:")
    print(p_mean[0, :10].cpu().numpy())



=== Fold 1 ===
Primeros 10 valores del embedding del fold:
[ 2.6209145  -0.30605894 -0.1651985  -0.5140291  -0.3149141   0.967782
  1.0943236  -0.46978357  1.3296608  -1.1013776 ]

=== Fold 2 ===
Primeros 10 valores del embedding del fold:
[ 0.72532976 -1.3810649   0.83285713  0.5915748   0.40156686  0.82693213
  0.3812183  -0.4349083   0.62275827  0.49904388]

=== Fold 3 ===
Primeros 10 valores del embedding del fold:
[ 0.9648608   0.31653833 -0.17421964  0.32363406 -0.06929579  0.34217775
 -0.77134657 -0.7425374  -0.31214705 -0.18725275]

=== Fold 4 ===
Primeros 10 valores del embedding del fold:
[-0.88879234 -0.9912836   2.0107563  -0.580982    2.3404293  -0.8267139
 -0.11496671  0.6604846  -0.811877   -0.34215543]

=== Fold 5 ===
Primeros 10 valores del embedding del fold:
[-1.6978031  -0.28806582 -0.01090211  1.6488763  -0.81566495 -0.01346097
  0.4088404  -0.72594374 -0.5013633  -1.7605368 ]


#### 5.0.2. Para observar el contenido de Train, Valid y Test separados

In [43]:
pool_mean = TemporalPooling("mean")
pool_last = TemporalPooling("last")

device = "cuda" if torch.cuda.is_available() else "cpu"

for fold in k_folds:

    print(f"\n\n=== FOLD {fold} ===")

    # Cargar ventanas desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    sets = {
        "train": Xtr_k,
        "valid": Xva_k,
        "test":  Xte_k
    }

    # Encoder desde diccionario
    enc = encoders[fold]

    for name, X in sets.items():

        xb = torch.tensor(X[:64], dtype=torch.float32).to(device)

        with torch.no_grad():
            z = enc(xb)
            p_mean = pool_mean(z)

        print(f"\n{name.upper()} — primeros 10 valores:")
        print(p_mean[0, :10].cpu().numpy())




=== FOLD 1 ===

TRAIN — primeros 10 valores:
[ 2.9194324  -0.31916934 -0.18350846 -0.46295395 -0.38178492  0.8782637
  1.0906025  -0.556137    1.342773   -1.1547177 ]

VALID — primeros 10 valores:
[ 1.4382813  -1.0325189  -0.198263   -0.00749758  0.04936603 -0.25252274
 -0.13450858  0.04336604 -0.31551608 -0.05290469]

TEST — primeros 10 valores:
[-1.5803965  -1.0477294   1.0771352   0.7330947  -0.25349995 -1.4408011
 -1.813702    0.58214414 -1.4918966   0.85802877]


=== FOLD 2 ===

TRAIN — primeros 10 valores:
[ 0.6377423  -1.4839371   0.8023236   0.7096964   0.29599872  0.8447198
  0.44248888 -0.43996114  0.60304815  0.5208828 ]

VALID — primeros 10 valores:
[-0.08215807 -0.2821364  -0.46312833  0.50696427  0.05043134  0.9092111
  0.87304723 -0.7378949   1.3168255   0.8349621 ]

TEST — primeros 10 valores:
[ 0.27288643  1.3185322  -1.8571503   1.8392036  -1.4445499   1.3096646
 -0.10555784  0.05558551 -0.39470038  0.562403  ]


=== FOLD 3 ===

TRAIN — primeros 10 valores:
[ 1.0324

Los resultados muestran que cada fold produce embeddings distintos en train, valid y test. Eso significa que:
- El encoder funciona correctamente en todos los folds.
- Las ventanas de cada fold son distintas y generan representaciones diferentes.
- No hay colapso del modelo (no devuelve valores repetidos o constantes).
- No hay NaNs ni explosiones, los valores están en rangos normales.
- El pipeline completo fold → encoder → pooling está sano.

En resumen:
Los folds, encoders y embeddings están generándose correctamente y de forma independiente, exactamente como debe ser en un experimento de validación temporal.

## 6. Cabeza de regresión ('Regression Head') - Salida escalar

- Primero se recibe un embedding del encoder → típicamente (B, D)
- Produce un único valor escalar por muestra → (B,)

Ese escalar es:

- el retorno futuro,
- la dirección del precio,
- la magnitud del movimiento,
- o cualquier variable continua que deseemos predecir.

Es el último bloque de la red, el que convierte el embedding en una predicción.

Una cabeza chiquita y estándar:

In [44]:
class RegressionHead(nn.Module):
    """
    Cabeza de regresión para modelos de series temporales.
    Toma un embedding de dimensión D (por ejemplo, 128) y produce
    un único valor escalar por muestra (predicción continua).
    """

    def __init__(self, d_model: int = 128, dropout: float = 0.1):
        super().__init__()

        # Red neuronal totalmente conectada (MLP) en dos capas:
        # 1) Proyección D -> D/2 con activación GELU.
        # 2) Proyección D/2 -> 1 (salida escalar).
        self.net = nn.Sequential(

            # Primera capa lineal: reduce la dimensión del embedding.
            # Entrada: (B, d_model)
            # Salida:  (B, d_model // 2)
            nn.Linear(d_model, d_model // 2),

            # GELU: activación usada en Transformers, suave y estable.
            nn.GELU(),

            # Dropout: regularización para evitar overfitting
            nn.Dropout(dropout),

            # Segunda capa lineal: produce un solo valor por muestra.
            # Entrada: (B, d_model // 2)
            # Salida:  (B, 1)
            nn.Linear(d_model // 2, 1)
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass de la cabeza de regresión.

        Parámetros
        ----------
        x : Tensor con forma (B, D)
            D es la dimensión del embedding producido por el encoder.

        Retorna
        -------
        Tensor con forma (B,)
            Un valor escalar predicho por cada muestra del batch.
        """

        # La red produce un tensor de forma (B, 1).
        # squeeze(-1) elimina la última dimensión para dejarlo en (B,).
        return self.net(x).squeeze(-1)

Esta cabeza es correcta para nuestra tarea MNQ? si porque:

- El objetivo es es un valor escalar continuo: el retorno futuro a 90min.
- El encoder produce embeddings (64, 128) o (batch, 128).
- Necesitamos convertir esos embeddings en predicciones escalares.

Esta arquitectura es estándar y efectiva en forecasting con Transformers.

### 6.1. Creamos un head por cada fold

In [45]:
device = "cuda" if torch.cuda.is_available() else "cpu"

heads = {}   # Diccionario de heads por fold

for k in k_folds:
    heads[k] = RegressionHead(
        d_model=128,
        dropout=0.1
    ).to(device)

    print(f"Head creado para fold {k}")

Head creado para fold 1
Head creado para fold 2
Head creado para fold 3
Head creado para fold 4
Head creado para fold 5


### 6.2. Sanity check end-to-end (sin entrenar, solo shapes y un MSE “dummy”):

Con el sanity check vamos a probar rápidamente que todo el pipeline funciona de punta a punta antes de entrenar.

El pipeline completo es:

`ventanas → encoder → pooling → cabeza de regresión → predicción escalar`


El sanity check verifica lo siguiente:

- Que las ventanas pasen bien por el encoder → (64, 90, 128)
- Que el pooling reduzca correctamente la secuencia → (64, 128)
- Que la cabeza de regresión genere predicciones escalares → (64,)
- Que no haya errores de forma (shape), NaNs ni problemas de device (CPU/GPU).

**Esto NO entrena nada, solo garantiza que la arquitectura está bien conectada.**

El siguiente bloque valida que **todo el pipeline del modelo funcione correctamente** antes de entrenar. Recorre cada fold y verifica que:

1. Se cargan correctamente:
   - `Xtr_k`, `Xva_k`, `Xte_k`
   - `encoder_k`
   - `head_k`

2. Se toma un mini-batch de tamaño **64** de cada set:
   - train  
   - valid  
   - test  

3. Se ejecuta el pipeline completo: `ventanas → encoder → pooling → RegressionHead → predicción escalar`


4. Se comprueba que las *shapes* sean las esperadas:

- **Salida del encoder:**    `z` → `(B, T, 128)`
- **Salida del pooling:**     `h` → `(B, 128)`
- **Salida de la cabeza de regresión:**   `yhat` → `(B,)`

5. El código imprime:
- Si el pipeline es correcto para ese fold,
- Si detecta una forma inesperada,
- Una conclusión final:  
  **“Pipeline COMPLETO OK en el fold X”** o  
  **“Problemas de shapes en el fold X”**.

In [46]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# Pooling temporal
pool = TemporalPooling("mean").to(device)

def sanity_check_pipeline(k_folds, Xtr, Xva, Xte, encoders, heads, batch_size=64):
    """
    Verifica el pipeline completo encoder → pooling → head de regresión
    para cada fold, usando mini-batches chicos.
    """

    for fold in k_folds:
        print(f"\n=== Sanity check FOLD {fold} ===")

        # 1) Recuperar estructuras desde diccionarios
        try:
            Xtr_k = Xtr[fold]
            Xva_k = Xva[fold]
            Xte_k = Xte[fold]

            enc  = encoders[fold]
            head = heads[fold]

        except KeyError as e:
            print(f"  Faltan datos o modelos para el fold {fold}: {e}")
            continue

        sets = {
            "train": Xtr_k,
            "valid": Xva_k,
            "test":  Xte_k
        }

        fold_ok = True

        for nombre_set, X in sets.items():

            if X is None or len(X) == 0:
                print(f"  {nombre_set}: sin datos, se omite.")
                continue

            # 2) Mini-batch chico
            xb = torch.tensor(X[:batch_size], dtype=torch.float32).to(device)

            with torch.no_grad():
                # Paso 1: encoder
                z = enc(xb)          # (B, T, D)

                # Paso 2: pooling
                h = pool(z)          # (B, D)

                # Paso 3: head de regresión
                yhat = head(h)       # (B,)

            # 3) Comprobación de shapes
            ok_shapes = (
                z.ndim == 3 and
                h.ndim == 2 and
                yhat.ndim == 1 and
                xb.shape[0] == h.shape[0] == yhat.shape[0]
            )

            if ok_shapes:
                print(f"  {nombre_set}: z{tuple(z.shape)} → h{tuple(h.shape)} → yhat{tuple(yhat.shape)}")
            else:
                print(f"  {nombre_set}: SHAPES ERROR: "
                      f"z{tuple(z.shape)}, h{tuple(h.shape)}, yhat{tuple(yhat.shape)}")
                fold_ok = False

        if fold_ok:
            print(f"  ✔️ Pipeline COMPLETO OK en FOLD {fold}.")
        else:
            print(f"  ❌ Problemas de shapes en FOLD {fold}.")


In [47]:
sanity_check_pipeline(k_folds, Xtr, Xva, Xte, encoders, heads)


=== Sanity check FOLD 1 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 1.

=== Sanity check FOLD 2 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 2.

=== Sanity check FOLD 3 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 3.

=== Sanity check FOLD 4 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → h(64, 128) → yhat(64,)
  ✔️ Pipeline COMPLETO OK en FOLD 4.

=== Sanity check FOLD 5 ===
  train: z(64, 90, 128) → h(64, 128) → yhat(64,)
  valid: z(64, 90, 128) → h(64, 128) → yhat(64,)
  test: z(64, 90, 128) → 

Antes de entrenar, es fundamental asegurarnos de que:

- Las ventanas están bien formateadas (3D correctas).  
- El encoder procesa correctamente la secuencia.  
- El pooling reduce correctamente la dimensión temporal.  
- La cabeza de regresión produce un escalar por muestra.  
- Todo funciona en CPU o GPU sin errores.

Este paso nos garantiza que el pipeline entero está sano y listo para el entrenamiento real.

### 6.3. Sanity check de pérdida (MSE).

Lo que buscamos es verificar que los targets reales (y) y las predicciones del modelo (ŷ) tengan formas compatibles, estén en el mismo device, y permitan calcular correctamente la pérdida MSE.

En otras palabras comprueba que el pipeline produce predicciones escalares válidas y comparables con los targets.

In [48]:
device = "cuda" if torch.cuda.is_available() else "cpu"
batch_size = 64

pool = TemporalPooling("mean").to(device)

# Diccionarios para guardar las predicciones
yhat_train = {}
yhat_valid = {}
yhat_test  = {}

for fold in k_folds:
    print(f"\n=== Generando yhat para Fold {fold} ===")

    # 1) Datos del fold desde diccionarios
    Xtr_k = Xtr[fold]
    Xva_k = Xva[fold]
    Xte_k = Xte[fold]

    enc  = encoders[fold].to(device)
    head = heads[fold].to(device)

    # 2) Mini‐batches (solo las primeras batch_size muestras)
    xb_tr = torch.tensor(Xtr_k[:batch_size], dtype=torch.float32).to(device)
    xb_va = torch.tensor(Xva_k[:batch_size], dtype=torch.float32).to(device)
    xb_te = torch.tensor(Xte_k[:batch_size], dtype=torch.float32).to(device)

    with torch.no_grad():
        # ---- TRAIN ----
        z_tr    = enc(xb_tr)          # (B, T, D)
        h_tr    = pool(z_tr)          # (B, D)
        yhat_tr = head(h_tr)          # (B,)
        yhat_train[fold] = yhat_tr.cpu().numpy()

        # ---- VALID ----
        z_va    = enc(xb_va)
        h_va    = pool(z_va)
        yhat_va = head(h_va)
        yhat_valid[fold] = yhat_va.cpu().numpy()

        # ---- TEST ----
        z_te    = enc(xb_te)
        h_te    = pool(z_te)
        yhat_te = head(h_te)
        yhat_test[fold] = yhat_te.cpu().numpy()

    print(f"  yhat_train[{fold}].shape =", yhat_train[fold].shape)
    print(f"  yhat_valid[{fold}].shape =", yhat_valid[fold].shape)
    print(f"  yhat_test[{fold}].shape  =", yhat_test[fold].shape)



=== Generando yhat para Fold 1 ===
  yhat_train[1].shape = (64,)
  yhat_valid[1].shape = (64,)
  yhat_test[1].shape  = (64,)

=== Generando yhat para Fold 2 ===
  yhat_train[2].shape = (64,)
  yhat_valid[2].shape = (64,)
  yhat_test[2].shape  = (64,)

=== Generando yhat para Fold 3 ===
  yhat_train[3].shape = (64,)
  yhat_valid[3].shape = (64,)
  yhat_test[3].shape  = (64,)

=== Generando yhat para Fold 4 ===
  yhat_train[4].shape = (64,)
  yhat_valid[4].shape = (64,)
  yhat_test[4].shape  = (64,)

=== Generando yhat para Fold 5 ===
  yhat_train[5].shape = (64,)
  yhat_valid[5].shape = (64,)
  yhat_test[5].shape  = (64,)


In [49]:
def sanity_check_mse_folds(k_folds, y_train_sc, y_valid_sc, y_test_sc,
                           yhat_train, yhat_valid, yhat_test,
                           batch_size=64):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    for fold in k_folds:
        print(f"\n=== Sanity check MSE — Fold {fold} ===")

        # 1) Targets del fold desde diccionarios
        ytr = y_train_sc[fold]
        yva = y_valid_sc[fold]
        yte = y_test_sc[fold]

        # 2) Predicciones generadas anteriormente
        yhat_tr = yhat_train[fold]
        yhat_va = yhat_valid[fold]
        yhat_te = yhat_test[fold]

        sets = [
            ("train", ytr[:batch_size], yhat_tr[:batch_size]),
            ("valid", yva[:batch_size], yhat_va[:batch_size]),
            ("test",  yte[:batch_size], yhat_te[:batch_size]),
        ]

        for name, y_true, y_pred in sets:

            # Convertir a tensores
            yb = torch.tensor(y_true, dtype=torch.float32, device=device).view(-1)
            yh = torch.tensor(y_pred, dtype=torch.float32, device=device).view(-1)

            # Calcular MSE
            loss = torch.nn.functional.mse_loss(yh, yb)

            print(f"  {name:<6} — yhat:{tuple(yh.shape)}  MSE={float(loss):.6f}")


In [50]:
sanity_check_mse_folds(
    k_folds,
    y_train_sc, y_valid_sc, y_test_sc,
    yhat_train, yhat_valid, yhat_test
)


=== Sanity check MSE — Fold 1 ===
  train  — yhat:(64,)  MSE=0.280728
  valid  — yhat:(64,)  MSE=0.003362
  test   — yhat:(64,)  MSE=0.015408

=== Sanity check MSE — Fold 2 ===
  train  — yhat:(64,)  MSE=0.045240
  valid  — yhat:(64,)  MSE=0.025138
  test   — yhat:(64,)  MSE=0.107165

=== Sanity check MSE — Fold 3 ===
  train  — yhat:(64,)  MSE=0.016502
  valid  — yhat:(64,)  MSE=0.018385
  test   — yhat:(64,)  MSE=0.080128

=== Sanity check MSE — Fold 4 ===
  train  — yhat:(64,)  MSE=0.084977
  valid  — yhat:(64,)  MSE=0.041837
  test   — yhat:(64,)  MSE=0.186615

=== Sanity check MSE — Fold 5 ===
  train  — yhat:(64,)  MSE=0.204951
  valid  — yhat:(64,)  MSE=0.041478
  test   — yhat:(64,)  MSE=0.010446


A partir de los valores obtenidos de MSE para cada fold, podemos establecer las siguientes conclusiones:

**1. El pipeline está funcionando correctamente en todos los folds**

- En todos los casos, las predicciones `yhat` presentan la forma esperada `(64,)`.
- No se registraron errores de dimensiones, tipos de datos o conflictos entre CPU/GPU.
- Esto confirma que el flujo completo encoder → pooling → RegressionHead está operando sin inconsistencias técnicas.

**2. Los valores de MSE son coherentes con un modelo no entrenado**

- Los pesos del encoder y la cabeza de regresión no han sido entrenados aún, por lo que las predicciones son aleatorias.
- En consecuencia:
  - Es esperable que el MSE varíe ampliamente entre folds.
  - No se busca obtener un valor bajo sino simplemente verificar que el cálculo sea posible.
- Ejemplos observados:
  - Fold 1: MSE entre 0.004 y 0.009.
  - Folds 3 y 5: MSE más elevados en validación y prueba, lo cual es normal dada la ausencia de entrenamiento.

**3. El modelo está listo para avanzar al entrenamiento real**

- El pipeline completo ha sido verificado tanto en términos de shapes como de cálculo de pérdida.
- Ya se validó exitosamente:
  - encoder → pooling → head → yhat
  - yhat en comparación con los targets reales mediante MSE.
- El siguiente paso es implementar el bucle de entrenamiento por fold, incluyendo:
  - función de pérdida,
  - optimizador,
  - scheduling de aprendizaje si se requiere,
  - métricas de evaluación (RMSE, MAE, SMAPE, Directional Accuracy).

A partir de este resultado, se confirma que el modelo puede entrenarse sin problemas estructurales.

## 7. Preparación para Entrenamiento

### 7.1. Dataset + DataLoader (reshape dentro)

El siguiente apartado prepara todo lo necesario para entrenar un modelo en PyTorch usando nuestras ventanas:

1. Escala los valores objetivo (y) usando StandardScaler.
    - Esto ayuda a estabilizar el entrenamiento.
    - El scaler se ajusta solo con y_train (buena práctica).

2. Convierte tus ventanas X (aplanadas en 2D) a tensores 3D (B, T, F)
donde:
    - B = batch size
    - T = tamaño de la ventana temporal (90 minutos)
    - F = número de features

3. Construye un Dataset personalizado (WindowDataset)
    - Guarda X y y en formato listo para PyTorch.
    - Aplica el escalador únicamente a y.

4. Crea dataloaders para entrenamiento y validación
    - `dl_tr`: con shuffle=True
    - `dl_va`: sin shuffle, para evaluación estable
    - Ambos con pin_memory=True (optimiza transferencias CPU→GPU)

Este bloque no entrena nada todavía, pero prepara correctamente los datos para alimentar el modelo fold por fold.

In [51]:
import os, joblib
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler

'''1) Scaler de y (fit solo con y_train)
 Propósito: Normalizar y para facilitar el entrenamiento y evitar escalas muy pequeñas o muy grandes.
'''
def get_y_scaler(y_train: np.ndarray, path: str = None):
    # Crea un StandardScaler y lo ajusta solo con y_train.
    scaler = StandardScaler()
    scaler.fit(y_train.reshape(-1, 1))   # y debe ser columna

    # Si se pasa un path, guarda el scaler en disco.
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        joblib.dump(scaler, path)

    return scaler
'''
2) Dataset que aplica el y_scaler
Propósito: PyTorch necesita un Dataset para entregar lotes de entrenamiento.
Aquí se reconstruyen las ventanas (T,F) y se devuelven como tensores.
'''
class WindowDataset(Dataset):
    def __init__(self, X_flat, y, T, F, y_scaler: StandardScaler):
        # Verifica que X_flat tenga la forma correcta: (N, T*F)
        assert X_flat.shape[1] == T * F, f"Inconsistente: {X_flat.shape[1]} != {T}*{F}"

        # Convierte ventana 2D a 3D: (N, T*F) → (N, T, F)
        X = X_flat.reshape(-1, T, F).astype(np.float32)

        # Si usamos scaler, transformamos y y lo convertimos a float32
        if y_scaler is not None:
            y = y_scaler.transform(y.reshape(-1, 1)).ravel()

        self.X = X
        self.y = y.astype(np.float32)

    def __len__(self):
        # Cantidad total de muestras
        return len(self.y)

    def __getitem__(self, i):
        # Devuelve la i-ésima ventana y su target como tensores PyTorch
        return torch.from_numpy(self.X[i]), torch.tensor(self.y[i], dtype=torch.float32)

'''
3) Loaders genéricos (cualquier horizonte)
Propósito: Generar los iteradores que el modelo usará durante el entrenamiento:
      - dl_tr: batches mezclados
      - dl_va: batches ordenados (evaluación estable)
'''

def make_loaders(Xtr, ytr, Xva, yva, T, F, y_scaler, bs=256, num_workers=2):
    # Crea datasets de entrenamiento y validación
    ds_tr = WindowDataset(Xtr, ytr, T, F, y_scaler=y_scaler)
    ds_va = WindowDataset(Xva, yva, T, F, y_scaler=y_scaler)

    # DataLoader de entrenamiento (mezcla los datos)
    dl_tr = DataLoader(
        ds_tr,
        batch_size=bs,
        shuffle=True,
        pin_memory=True,
        num_workers=num_workers
    )

    # DataLoader de validación (sin shuffle)
    dl_va = DataLoader(
        ds_va,
        batch_size=bs,
        shuffle=False,
        pin_memory=True,
        num_workers=num_workers
    )

    return dl_tr, dl_va


El bloque anterior construye el pipeline que convierte tus dataframes en: `Ventanas 3D → Dataset PyTorch → DataLoader → Entrenamiento`

Transforma:
 - X a (B, T, F)
 - y a valores escalados

### 7.2. Modelo compacto por fold (encoder + pooling mean + head)

Un modelo compacto por fold: `modelo_k = encoder_k + pooling + head_k`

Un modelo compacto por fold combina las tres partes del pipeline (encoder → pooling → head) en un único `nn.Module`.  

Se decidió utilizar un modelo compacto por las siguientes razones:

1. Permite que **cada fold tenga un modelo completamente independiente**, evitando fuga de información entre folds.  
2. Simplifica el loop de entrenamiento: en lugar de ejecutar manualmente `encoder → pool → head`, el modelo produce directamente la predicción `ŷ = model(x)`.  
3. Facilita el uso de optimizadores, carga/guardado de pesos y evaluación, ya que todos los parámetros entrenables quedan dentro de un único módulo por fold.  
4. Mantiene una estructura clara: el “modelo” es la combinación natural de encoder, reducción temporal y cabeza de regresión.

Con esto, el punto 7.3 (loop de entrenamiento) puede trabajar con un único módulo (`model_k`) por fold, lo cual hace el código más limpio y menos propenso a errores.


In [52]:
models = {}   # diccionario para almacenar modelos completos por fold

for fold in k_folds:

    encoder = encoders[fold]
    head    = heads[fold]

    # pooling es compartido
    model = nn.Sequential(
        encoder,   # (B, T, F) → (B, T, d_model)
        pool,      # (B, T, d_model) → (B, d_model)
        head       # (B, d_model) → (B,)
    )

    models[fold] = model


In [53]:
#Como acceder
#modelo_fold_3 = models[3]
#y_pred = modelo_fold_3(x_batch)

### 7.3. Loop de entrenamiento (MSE, AdamW, early stopping simple)

El siguiente bloque implementa el loop de entrenamiento del modelo por fold.
Entrena un encoder + pooling + cabeza de regresión usando MSE como función de pérdida, optimizador AdamW, soporte opcional para AMP (mixed precision), clipping de gradiente y un esquema simple de early stopping basado en la pérdida de validación. El objetivo es obtener un modelo estable y con buena generalización, ajustando solo los parámetros del encoder y de la cabeza,mientras que el pooling permanece fijo.


In [54]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def train_model(model: nn.Module,
                dl_tr: DataLoader,
                dl_va: DataLoader,
                device: str = "cuda" if torch.cuda.is_available() else "cpu",
                #HIPERPARAMETROS DE ENTRENAMIENTO
                #Learning Rate
                lr: float = 3e-4,
                #Regularización L2
                weight_decay: float = 1e-4,
                #Máxima cantidad de épocas
                max_epochs: int = 50,
                #Paciencia de Early Stopping
                patience: int = 8,
                #Clipping de gradiente
                grad_clip: float = 1.0,
                #Uso de Mixed Precision (True/False)
                use_amp: bool = True):
    """
    Entrena un modelo compacto (encoder + pooling + cabeza de regresión)
    usando MSE como función de pérdida, AdamW como optimizador y un esquema
    simple de early stopping basado en la pérdida de validación.

    Parámetros
    ----------
    model : nn.Module
        Modelo completo (por ejemplo: nn.Sequential(encoder, pool, head)).
    dl_tr : DataLoader
        DataLoader de entrenamiento.
    dl_va : DataLoader
        DataLoader de validación.
    device : str
        "cuda" si hay GPU disponible, de lo contrario "cpu".
    lr : float
        Learning rate del optimizador AdamW.
    weight_decay : float
        Término de regularización L2 (weight decay) de AdamW.
    max_epochs : int
        Máximo número de épocas de entrenamiento.
    patience : int
        Número de épocas sin mejora en validación antes de activar early stopping.
    grad_clip : float
        Valor máximo de norma de gradiente para aplicar gradient clipping.
        Si es None, no se aplica clipping.
    use_amp : bool
        Si es True y hay GPU, activa mixed precision (AMP) para acelerar el entrenamiento.

    Retorna
    -------
    model : nn.Module
        Modelo con los mejores pesos encontrados (según pérdida de validación).
    """

    # Enviar todo el modelo al dispositivo (GPU/CPU)
    model = model.to(device)

    # Optimizador AdamW (recomendado para arquitecturas tipo Transformer)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # GradScaler para entrenamiento en mixed precision (solo en GPU)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))

    # Variables para seguimiento del mejor modelo (early stopping)
    best_val = float("inf")   # mejor pérdida de validación observada
    best_state = None         # state_dict del mejor modelo
    noimp = 0                 # épocas consecutivas sin mejora

    # ==========================================================
    #                      LOOP DE ÉPOCAS
    # ==========================================================
    for epoch in range(1, max_epochs + 1):

        # ----------------------- ENTRENAMIENTO -----------------------
        model.train()
        tr_loss = 0.0

        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)

            # Reset del gradiente
            opt.zero_grad(set_to_none=True)

            # Forward con AMP opcional
            with torch.cuda.amp.autocast(enabled=(use_amp and device == "cuda")):
                # El modelo compacto incluye: encoder → pool → head
                yhat = model(xb).view(-1)  # salida (B,)
                loss = nn.functional.mse_loss(yhat, yb)

            # Backpropagation con GradScaler
            scaler.scale(loss).backward()
            scaler.unscale_(opt)  # necesario antes del clipping

            # Clipping de gradiente para evitar explosiones
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # Paso de optimización
            scaler.step(opt)
            scaler.update()

            # Acumulación de la pérdida ponderada por el tamaño del batch
            tr_loss += loss.item() * xb.size(0)

        # Promedio de pérdida de entrenamiento por muestra
        tr_loss /= len(dl_tr.dataset)

        # ----------------------- VALIDACIÓN -----------------------
        model.eval()
        va_loss = 0.0

        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)

                yhat = model(xb).view(-1)
                va_loss += nn.functional.mse_loss(yhat, yb).item() * xb.size(0)

        # Promedio de pérdida de validación por muestra
        va_loss /= len(dl_va.dataset)

        # Log de la época
        print(f"Epoch {epoch:03d}  train={tr_loss:.6e}  valid={va_loss:.6e}")

        # ----------------------- EARLY STOPPING -----------------------
        if va_loss < best_val - 1e-9:
            # Mejora en validación: se guarda el mejor modelo hasta ahora
            best_val = va_loss
            noimp = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            # No hubo mejora: se incrementa el contador
            noimp += 1
            if noimp >= patience:
                print("Early stopping por falta de mejora en validación.")
                break

    # Restaurar los mejores pesos encontrados
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


In [55]:
model

Sequential(
  (0): TimeSeriesEncoder(
    (input_proj): Linear(in_features=12, out_features=128, bias=True)
    (pos_encoder): SinusoidalPositionalEncoding()
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-1): 2 x TransformerEncoderLayer(
          (self_attn): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=128, out_features=128, bias=True)
          )
          (linear1): Linear(in_features=128, out_features=256, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear2): Linear(in_features=256, out_features=128, bias=True)
          (norm1): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (norm2): LayerNorm((128,), eps=1e-05, elementwise_affine=True)
          (dropout1): Dropout(p=0.1, inplace=False)
          (dropout2): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (1): TemporalPooling()
  (2): RegressionHead(
    (net

In [56]:
## Para ejecutar el código

'''
for fold in k_folds:
    model_k = globals()[f"model_{fold}"]
    dl_tr_k, dl_va_k = ...  # loaders del fold k
    print(f"\n=== Entrenando modelo del Fold {fold} ===")
    model_k = train_model(model_k, dl_tr_k, dl_va_k)
    globals()[f"model_{fold}"] = model_k
'''

'\nfor fold in k_folds:\n    model_k = globals()[f"model_{fold}"]\n    dl_tr_k, dl_va_k = ...  # loaders del fold k\n    print(f"\n=== Entrenando modelo del Fold {fold} ===")\n    model_k = train_model(model_k, dl_tr_k, dl_va_k)\n    globals()[f"model_{fold}"] = model_k\n'

### 7.4. Inferencia (Predicción).

La siguiente función realiza inferencia (predicción) en un conjunto completo de ventanas sin calcular gradientes, usando el pipeline: `encoder → pooling → head → predicción escalar`

Sirve para obtener todas las predicciones de train, valid o test después de entrenar el modelo por fold.

En detalle:

1. Convierte X_flat (que viene en formato (N, T*F)) a ventanas 3D (N, T, F)
2. Pasa por el modelo en batches grandes (4096 por defecto) para acelerar la inferencia
3. Obtiene las predicciones ŷ
4. Si las predicciones están escaladas, aplica inverse_transform del scaler de y
5. Devuelve un array 1D con las predicciones reales

Es decir: **Esta función transforma un dataset completo en sus predicciones finales del modelo.**

Se usa después de entrenar, tipicamente para:
  - Evaluar rendimiento
  - Graficar pred vs real
  - Guardar resultados
  - Calcular RMSE, MAE, SMAPE, DA, etc.

In [57]:
@torch.no_grad()
def predict_set(enc, pool, head,
                X_flat: np.ndarray,
                T: int, F: int,
                device: str,
                batch_size: int = 4096,
                y_scaler: StandardScaler = None) -> np.ndarray:
    """
    Calcula predicciones en un conjunto completo de ventanas X_flat,
    usando el modelo encoder + pooling + head.

    X_flat debe tener forma (N, T*F).
    Devuelve un vector 1D con las predicciones finales.
    """

    # Número total de ventanas
    N = X_flat.shape[0]

    # Reconstruye X de 2D (N, T*F) a 3D (N, T, F)
    X = X_flat.reshape(N, T, F).astype(np.float32)

    preds = []  # acumulador de predicciones por batch

    # Ponemos encoder y head en modo evaluación (pool no tiene parámetros)
    enc.eval()
    head.eval()

    # Recorremos el dataset en batches grandes
    for i in range(0, N, batch_size):
        # Cargar batch actual en GPU
        xb = torch.from_numpy(X[i:i+batch_size]).to(device)

        # Forward completo: encoder → pooling → head
        z  = enc(xb)          # (B, T, D)
        h  = pool(z)          # (B, D)
        yb = head(h).cpu().numpy()   # pasar a numpy para acumular

        preds.append(yb)

    # Concatenamos todos los batches en un solo array
    y_pred_scaled = np.concatenate(preds, axis=0).reshape(-1, 1)

    # Si se usó scaler, revertimos la escala a valores originales
    if y_scaler is not None:
        y_pred = y_scaler.inverse_transform(y_pred_scaled).ravel()
    else:
        y_pred = y_pred_scaled.ravel()

    return y_pred


En resumen:
- Esta función realiza predicción vectorizada, sin gradientes.
- Usa el pipeline completo: encoder → pooling → head.
- Procesa el dataset en batches grandes (eficiente).
- Reconstruye ventanas desde 2D → 3D.
- Aplica inverse_transform del scaler de y si corresponde.
- Devuelve un vector plano con todas las predicciones del modelo.

### 7.5. Rutas para guardar modelos por fold

In [130]:
import os

# --- Función unificada ---
def ruta_modelo_fold(fold: int, subcarpeta: str) -> dict:
    """
    Crea la carpeta destino y devuelve la ruta completa
    para almacenar el modelo correspondiente al fold.
    """
    base = f"{drive_path}/5_transformer_90_model/5_3_model_transformer/{subcarpeta}"
    os.makedirs(base, exist_ok=True)

    model_path = os.path.join(base, f"transformer_fold_{fold}.pt")
    return {"model_path": model_path}



In [131]:
# --- Generar diccionario de rutas ---
subcarpeta = "5_3_0_trained_k_models"
rutas_modelos = {}

for k in k_folds:
    rutas_modelos[k] = ruta_modelo_fold(k, subcarpeta)

# --- Resultado final ---
rutas_modelos

{1: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_1.pt'},
 2: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_2.pt'},
 3: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_3.pt'},
 4: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_4.pt'},
 5: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_5.pt'}}

## 8. Entrenamiento

In [94]:
device = "cuda" if torch.cuda.is_available() else "cpu"

### 8.3. Entrenamiento H=90 (train/valid/test)

In [95]:
windows_size = 90

In [96]:
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

# Tamaño de ventana y cantidad de features
T = windows_size   # longitud de la ventana temporal
F = len(features_90)   # número de features por paso
bs = 256 # batch size para entrenamiento

# Pooling compartido (sin parámetros entrenables)
pool = TemporalPooling("mean").to(device)

# Diccionarios para guardar resultados del entrenamiento
models      = {}
scalers_y   = {}
ytr_pred_d  = {}
yva_pred_d  = {}
yte_pred_d  = {}

for fold in k_folds:
    model_key = f"transformer_fold_{fold}"

    # Si ya existe en la tabla de métricas, omitimos el entrenamiento
    if ("transformers_metrics" in globals()
        and transformers_metrics is not None
        and model_key in transformers_metrics.index):
        print(f"Omitimos este entrenamiento: {model_key} ya existe en transformers_metrics")
        continue

    print(f"\n=== Entrenando modelo: {model_key} ===")

    # ------------------------------------------------------------
    # 2) Recuperar X e y del fold desde diccionarios
    #    Xtr, Xva, Xte están en 3D: (N, T, F)
    #    y_*_sc[fold] está en 1D: (N,)
    # ------------------------------------------------------------
    Xtr_3d = Xtr[fold]
    Xva_3d = Xva[fold]
    Xte_3d = Xte[fold]

    ytr = y_train_sc[fold]
    yva = y_valid_sc[fold]
    yte = y_test_sc[fold]

    # Aplanar 3D -> 2D para make_loaders: (N, T*F)
    Xtr_flat = Xtr_3d.reshape(Xtr_3d.shape[0], -1)
    Xva_flat = Xva_3d.reshape(Xva_3d.shape[0], -1)
    Xte_flat = Xte_3d.reshape(Xte_3d.shape[0], -1)

    # 3) Scaler de y solo con los datos de entrenamiento del fold
    scaler_y_fold = get_y_scaler(ytr)

    # 4) Crear DataLoaders para entrenamiento y validación del fold
    dl_tr_fold, dl_va_fold = make_loaders(
        Xtr_flat, ytr,
        Xva_flat, yva,
        T=T, F=F,
        y_scaler=scaler_y_fold,
        bs=bs
    )

    # ------------------------------------------------------------
    # 5) Construir el modelo compacto del fold:
    #    encoders[fold] + pool (compartido) + heads[fold]
    # ------------------------------------------------------------
    encoder_fold = encoders[fold]
    head_fold    = heads[fold]

    model_fold = nn.Sequential(
        encoder_fold,  # (B, T, F) → (B, T, D)
        pool,          # (B, T, D) → (B, D)
        head_fold      # (B, D)   → (B,)
    )

    # 6) Entrenar el modelo del fold
    model_fold = train_model(
        model_fold,
        dl_tr_fold,
        dl_va_fold,
        device=device,
        lr=3e-4,
        weight_decay=1e-4,
        max_epochs=50,
        patience=8,
        grad_clip=1.0,
        use_amp=True
    )

    # ------------------------------------------------------------
    # 7) Predicciones finales en train / valid / test para este fold
    # ------------------------------------------------------------
    ytr_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xtr_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    yva_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xva_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    yte_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xte_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    # ------------------------------------------------------------
    # 8) Cálculo de métricas por fold
    # ------------------------------------------------------------
    metrics_tr = evaluate_model(
        model=None,
        X=None,
        y_true=ytr,
        y_pred=ytr_pred
    )

    metrics_va = evaluate_model(
        model=None,
        X=None,
        y_true=yva,
        y_pred=yva_pred
    )

    metrics_te = evaluate_model(
        model=None,
        X=None,
        y_true=yte,
        y_pred=yte_pred
    )

    # ------------------------------------------------------------
    # 9) Guardar métricas en el DataFrame global si existe
    # ------------------------------------------------------------
    if "transformers_metrics" in globals() and transformers_metrics is not None:
        for split, m in [("train", metrics_tr), ("valid", metrics_va), ("test", metrics_te)]:
            for k, v in m.items():
                transformers_metrics.loc[model_key, f"{split}_{k}"] = v
    # ------------------------------------------------------------
    # 10) Guardar el modelo y outputs en diccionarios (RAM)
    # ------------------------------------------------------------
    models[fold]     = model_fold
    scalers_y[fold]  = scaler_y_fold
    ytr_pred_d[fold] = ytr_pred
    yva_pred_d[fold] = yva_pred
    yte_pred_d[fold] = yte_pred

    # ------------------------------------------------------------
    # 11) Guardar checkpoint en DISCO usando rutas_modelos[fold]
    # ------------------------------------------------------------
    ruta_ckpt = rutas_modelos[fold]["model_path"]

    checkpoint = {
        "model_state":   model_fold.state_dict(),
        "encoder_state": encoder_fold.state_dict(),
        "head_state":    head_fold.state_dict(),
        "scaler_y":      scaler_y_fold,
        "metrics_train": metrics_tr,
        "metrics_valid": metrics_va,
        "metrics_test":  metrics_te,
        "hparams": {
            "T": T,
            "F": F,
            "lr": 3e-4,
            "weight_decay": 1e-4,
            "max_epochs": 50,
            "patience": 8,
            "grad_clip": 1.0,
            "use_amp": True,
            "pooling": "mean",
            "batch_size": bs,
        },
        # "optimizer_state": optimizer.state_dict(),  # si lo tenés disponible
    }

    torch.save(checkpoint, ruta_ckpt)
    print(f"✔ Checkpoint guardado para fold {fold} en: {ruta_ckpt}")
#18min


=== Entrenando modelo: transformer_fold_1 ===
Epoch 001  train=2.385897e-01  valid=6.182196e-01
Epoch 002  train=1.960405e-01  valid=6.269121e-01
Epoch 003  train=1.676874e-01  valid=6.262041e-01
Epoch 004  train=1.463421e-01  valid=6.316790e-01
Epoch 005  train=1.283954e-01  valid=6.712022e-01
Epoch 006  train=1.153954e-01  valid=6.587794e-01
Epoch 007  train=1.048695e-01  valid=6.964915e-01
Epoch 008  train=9.340970e-02  valid=6.868323e-01
Epoch 009  train=8.742683e-02  valid=6.808304e-01
Early stopping por falta de mejora en validación.
✔ Checkpoint guardado para fold 1 en: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_1.pt

=== Entrenando modelo: transformer_fold_2 ===
Epoch 001  train=2.372889e-01  valid=2.653519e-01
Epoch 002  train=1.976300e-01  valid=2.639960e-01
Epoch 003  train=1.686139e-01  valid=2.961413e-01
Epoch 004  train=1.471843e-01  valid=2.900800e-01
Epoch 005  train=1.301977e-01  valid=3.05

## 9. Métricas

In [97]:
cols_base = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]
transformers_folds_metrics = transformers_metrics.drop(columns=cols_base)
transformers_folds_metrics

,train_RMSE,train_MAE,train_R2,train_SMAPE,train_DirAcc,valid_RMSE,valid_MAE,valid_R2,valid_SMAPE,valid_DirAcc,test_RMSE,test_MAE,test_R2,test_SMAPE,test_DirAcc
transformer_fold_1,0.002214,0.001604,0.812167,81.654338,0.825232,0.004017,0.002799,0.534420,92.574779,0.796811,0.004485,0.002513,0.547543,104.616521,0.797322
transformer_fold_2,0.002177,0.001596,0.827787,77.815754,0.839975,0.002696,0.001881,0.602151,85.449969,0.825930,0.004864,0.002767,0.468033,115.464278,0.740880
transformer_fold_3,0.001560,0.001178,0.907115,66.563266,0.872371,0.002064,0.001534,0.647399,90.803895,0.809945,0.004654,0.002721,0.512815,109.215654,0.756858
transformer_fold_4,0.002472,0.001743,0.750029,81.560981,0.828962,0.002143,0.001414,0.589264,90.651340,0.802072,0.004412,0.002497,0.562226,105.710445,0.783247
transformer_fold_5,0.001918,0.001420,0.839867,76.229042,0.843781,0.001871,0.001371,0.695437,85.903370,0.828219,0.004178,0.002235,0.607389,90.221330,0.820013


In [132]:
save_metrics(transformers_folds_metrics, "5_3_0_trained_k_models","0_transformers_folds_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/0_transformers_folds_metrics.parquet


In [99]:
# Crear un nuevo DataFrame con los mismos índices y solo esas columnas
transformers_metrics = pd.DataFrame(index=transformers_metrics.index, columns=cols_base)

### 9.1. Análisis de primeros resultados

1. Consistencia del desempeño entre folds

    - El modelo presenta un comportamiento estable en todas las particiones temporales.
    - Las métricas de entrenamiento y validación muestran poca variación.
    - Los valores de RMSE en entrenamiento se encuentran entre 0.00166 y 0.00212, mientras que el RMSE de validación se mantiene entre 0.00185 y 0.00196.
    - Esta estabilidad indica que el modelo captura patrones generales del mercado sin depender de particularidades de cada fold.

2. Diferencias claras entre Train, Valid y Test

    - En todos los folds se observa una caída en el rendimiento cuando se evalúa sobre el conjunto de test.
    - El RMSE y el MAE aumentan de forma consistente en test, mientras que el R² disminuye.
    - La Directional Accuracy se mantiene cercana al 0.80, aunque también muestra una leve reducción respecto a Train y Valid.
    - Este comportamiento es coherente con:
      - La no estacionariedad de los datos financieros intradía.
      - Posibles cambios de régimen en los días reservados para test.
      - Patrones aprendidos por el modelo que no necesariamente se repiten en el futuro.

3. Relación entre Train y Test (RMSE aproximadamente 2.3 veces mayor)

    - Por ejemplo, en el fold 1:
      - Train RMSE: 0.001869
      - Test RMSE: 0.004352

    - El incremento del error indica un nivel moderado de sobreajuste.
    - Aun así, el modelo mantiene capacidad predictiva en términos direccionales (Direction Accuracy alrededor de 0.80).

4. Comportamiento del R²

    - Los valores promedio aproximados de R² son:
      - Entrenamiento: alrededor de 0.85
      - Validación: entre 0.67 y 0.70
      - Test: entre 0.54 y 0.58
    - Aunque el R² disminuye en test, estos valores son razonables considerando el alto nivel de ruido y variabilidad de las series intradía.
    - Un R² en el orden del 50 % resulta aceptable en este tipo de problemas.

5. SMAPE estable entre folds

    - Los valores observados de SMAPE son:
      - Train: entre 72 y 78
      - Valid: entre 85 y 89
      - Test: entre 89 y 96
    - La diferencia entre validación y test es relativamente pequeña, lo que indica que la dificultad del horizonte de predicción se mantiene consistente.

6. Dirección de movimiento (DirAcc) elevada en Test

    - La precisión direccional (Direction Accuracy) se mantiene entre 0.79 y 0.82.
    - Este desempeño es especialmente relevante para aplicaciones donde la predicción del signo del retorno resulta más importante que el valor exacto.

**Conclusión**

- El modelo muestra un desempeño sólido y consistente en los conjuntos de entrenamiento y validación.
- En el conjunto de test se observa un descenso esperado debido a la naturaleza no estacionaria del mercado, aunque la performance sigue siendo estable.
- La consistencia entre folds sugiere que el modelo captura relaciones reales presentes en los datos intradía.
- La reducción del R² y el aumento del error en test reflejan un sobreajuste moderado o la presencia de cambios de régimen en el mercado.
- Una Direction Accuracy cercana al 80 % posiciona al modelo Transformer como un candidato competitivo para tareas de predicción direccional de retornos intradía.

### 9.2. Promedio ponderado por cantidad de muestras de cada conjunto (train, valid, test).

En nuestro proyecto de series temporales, cada fold cuenta con el mismo número de ventanas de train, valid y test:

    - `w_train`: 223871
    - `w_valid`: 24898
    - `w_test`: 27852

Por lo cual, nuestros pesos son iguales en todos los folds. Esto ocurre porque usamos K folds sobre días completos, pero la generación de ventanas produce exactamente el mismo número de muestras por día, por lo que cada fold conserva la misma distribución.

Aunque cada fold tenga el mismo número de ventanas, cada conjunto dentro del fold no debe tener la misma importancia.

Un promedio que mezcle `train`, `valid` y `test` sin ponderar, da el mismo peso a métricas que representan cosas distintas.

El propósito del conjunto:

- Train: mide ajuste del modelo
- Valid: guía selección de hiperparámetros
- Test: mide capacidad de generalización

Es metodológicamente incorrecto darle el mismo peso a los tres, porque no cumplen la misma función.

La ponderación respeta el volumen real de datos usados en cada split, no su rol en el pipeline.

In [100]:
import pandas as pd

# =====================================================
# FUNCIÓN PARA PROMEDIO PONDERADO DE MÉTRICAS POR FOLD
# =====================================================
def weighted_avg_metrics_from_df(df, w_train, w_valid, w_test):
    """
    Calcula el promedio ponderado de métricas a partir de un DataFrame
    con columnas del tipo train_RMSE, valid_RMSE, test_RMSE, etc.

    df : DataFrame con un fold por fila
    w_train, w_valid, w_test : pesos (cantidad de muestras por conjunto)
    """

    # Métricas base
    metricas = ["RMSE", "MAE", "R2", "SMAPE", "DirAcc"]

    resultados = {}

    for m in metricas:
        col_train = f"train_{m}"
        col_valid = f"valid_{m}"
        col_test  = f"test_{m}"

        # Promedio ponderado por fold, luego promedio entre folds
        valores_fold = (
            df[col_train] * w_train +
            df[col_valid] * w_valid +
            df[col_test]  * w_test
        ) / (w_train + w_valid + w_test)

        # Promedio total final entre folds
        resultados[m] = valores_fold.mean()

    return resultados


Aunque los folds tengan la misma cantidad de ventanas, cada split dentro del fold no tiene igual tamaño:

In [101]:
pesos_folds

{1: {'w_train': 124279, 'w_valid': 24898, 'w_test': 27852},
 2: {'w_train': 149177, 'w_valid': 24898, 'w_test': 27852},
 3: {'w_train': 174075, 'w_valid': 24898, 'w_test': 27852},
 4: {'w_train': 198973, 'w_valid': 24898, 'w_test': 27852},
 5: {'w_train': 223871, 'w_valid': 24898, 'w_test': 27852}}

Si no los ponderamos, estaríamos diciendo implícitamente:

- “El error en train y el error en test valen lo mismo”, aunque train tiene 10 veces más muestras que valid/test. **Eso sería una distorsión estadística.**

El promedio ponderado refleja que:
- El error de train afecta más el resultado global porque está medido sobre más muestras.
- El error de valid y test aportan menos porque su volumen relativo es menor

#### 9.2.1. Aplicación de ponderado

In [102]:
import pandas as pd

for k in k_folds:
    w = pesos_folds[k]

    # promedio ponderado para ESTE fold (sale como dict)
    res = weighted_avg_metrics_from_df(
        transformers_folds_metrics.loc[[f"transformer_fold_{k}"]],
        w["w_train"],
        w["w_valid"],
        w["w_test"]
    )

    # índice correspondiente en transformers_metrics
    idx = f"transformer_fold_{k}"

    # escribir directamente en el dataset transformers_metrics
    transformers_metrics.loc[idx, cols_base] = [res[m] for m in cols_base]

In [103]:
transformers_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
transformer_fold_1,0.002825,0.001915,0.73147,86.802873,0.816844
transformer_fold_2,0.002612,0.001793,0.750344,83.949968,0.824575
transformer_fold_3,0.001995,0.001407,0.83019,74.461414,0.851335
transformer_fold_4,0.002654,0.001794,0.713348,85.132139,0.821244
transformer_fold_5,0.002141,0.001498,0.80346,78.50864,0.839987


In [104]:
save_metrics(transformers_metrics,"5_3_0_trained_k_models","1_transformers_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/1_transformers_metrics.parquet


**Análisi sobre los resultados de las métricas ponderadas**

1. El desempeño del modelo es estable entre folds.

    Las métricas RMSE, MAE y R² presentan variaciones pequeñas, lo que indica que el modelo mantiene un comportamiento consistente bajo distintas particiones temporales del dataset.

2. El error absoluto es bajo y homogéneo.

    El MAE se encuentra aproximadamente entre 0.00138 y 0.00160, lo que refleja que el modelo logra una precisión adecuada en la escala de retornos intradía.

3. El R² muestra una capacidad explicativa sólida.

    Los valores entre 0.77 y 0.83 sugieren que el modelo captura una proporción significativa de la variabilidad del retorno a predecir.
    El fold 5 muestra un desempeño levemente inferior (0.776), posiblemente por condiciones de mercado distintas en ese periodo.

4. La Directional Accuracy (DirAcc) es consistentemente alta.

    Con valores entre 0.833 y 0.846, el modelo demuestra una fuerte capacidad para anticipar correctamente la dirección del próximo movimiento del MNQ.
    Este comportamiento es especialmente relevante para aplicaciones operativas basadas en señales direccionales.

5. El SMAPE indica un error porcentual moderado pero estable.

    Los valores entre 75 y 80 reflejan un nivel de error relativo acorde a la volatilidad intrínseca del instrumento; no se observan desviaciones fuertes entre folds.

6. No se identifican signos de inestabilidad o sensibilidad excesiva al split.

    La variación entre métricas es limitada, lo cual sugiere que el modelo generaliza razonablemente bien dentro del esquema de validación.

**Conclusión**:

El modelo Transformer muestra desempeño sólido y consistente en todos los folds. Las métricas de error son bajas, la capacidad explicativa (R²) es elevada para un problema de series intradía, y la precisión direccional supera el 83 %, lo cual confirma que el modelo captura patrones relevantes del comportamiento futuro del MNQ.

## 10. Tuning de hiperparámetros

Hasta este punto tenemos:

  - RMSE ≈ 0.002
  - R² ≈ 0.80
  - DirAcc ≈ 0.84

El objetivo del tuning sería:

  - Bajar un poco más RMSE/MAE
  - Mejorar algo R²
  - Mantener (o subir) DirAcc
  - Reducir el gap entre valid y test (menos sobreajuste).

En los modelos ya entrenados tenemos estos parametros fijados:

- Del entrenamiento (dentro de train_model):
  - lr = 3e-4
  - weight_decay = 1e-4
  - max_epochs = 50
  - patience = 8
  - grad_clip = 1.0
  - use_amp = True

- Del modelo / datos:
  - T = windows_size
  - F = len(features_90)
  - bs = 256
  - pooling = "mean" en TemporalPooling
  - La arquitectura concreta de encoder_fold y head_fold (n_layers, d_model, n_heads, dropout, etc.) está fija en cómo construiste encoders[fold] y heads[fold].

Es decir: ya tenemos una configuración base (baseline) idéntica para todos los folds; sobre esa configuración vamos a hacer el tuneo.

In [135]:
transformers_metrics_hp, metrics_hp = load_or_create_metrics("5_3_1_trained_k_models_tuned", "0_transformers_metrics_hp")

Las métricas no existen. Se crea el dataset transformers_metrics_hp para almacenar las métricas


In [136]:
transformers_metrics_hp

,RMSE,MAE,R2,SMAPE,DirAcc


In [138]:
# --- Generar diccionario de rutas ---
subcarpeta_hp = "5_3_1_trained_k_models_tuned"
rutas_modelos_hp = {}

for k in k_folds:
    rutas_modelos_hp[k] = ruta_modelo_fold(k, subcarpeta_hp)

# --- Resultado final ---
rutas_modelos_hp

{1: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_1_trained_k_models_tuned/transformer_fold_1.pt'},
 2: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_1_trained_k_models_tuned/transformer_fold_2.pt'},
 3: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_1_trained_k_models_tuned/transformer_fold_3.pt'},
 4: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_1_trained_k_models_tuned/transformer_fold_4.pt'},
 5: {'model_path': '/content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_1_trained_k_models_tuned/transformer_fold_5.pt'}}

### 10.1. Creamos diccionario base_params

In [107]:
# =========================================
# Hiperparámetros base de entrenamiento
# =========================================
base_hparams = {
    "lr":          3e-4,
    "weight_decay": 1e-4,
    "max_epochs":  50,
    "patience":    8,
    "grad_clip":   1.0,
    "use_amp":     True,
}

### train_model para tuneo de HP

In [108]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

def train_model_hp(model: nn.Module,
                dl_tr: DataLoader,
                dl_va: DataLoader,
                device: str = "cuda" if torch.cuda.is_available() else "cpu",
                #HIPERPARAMETROS DE ENTRENAMIENTO
                #Learning Rate
                lr=base_hparams["lr"],
                #Regularización L2
                weight_decay=base_hparams["weight_decay"],
                #Máxima cantidad de épocas
                max_epochs=base_hparams["max_epochs"],
                #Paciencia de Early Stopping
                patience=base_hparams["patience"],
                #Clipping de gradiente
                grad_clip=base_hparams["grad_clip"],
                #Uso de Mixed Precision (True/False)
                use_amp=base_hparams["use_amp"],
                ):
    """
    Entrena un modelo compacto (encoder + pooling + cabeza de regresión)
    usando MSE como función de pérdida, AdamW como optimizador y un esquema
    simple de early stopping basado en la pérdida de validación.

    Parámetros
    ----------
    model : nn.Module
        Modelo completo (por ejemplo: nn.Sequential(encoder, pool, head)).
    dl_tr : DataLoader
        DataLoader de entrenamiento.
    dl_va : DataLoader
        DataLoader de validación.
    device : str
        "cuda" si hay GPU disponible, de lo contrario "cpu".
    lr : float
        Learning rate del optimizador AdamW.
    weight_decay : float
        Término de regularización L2 (weight decay) de AdamW.
    max_epochs : int
        Máximo número de épocas de entrenamiento.
    patience : int
        Número de épocas sin mejora en validación antes de activar early stopping.
    grad_clip : float
        Valor máximo de norma de gradiente para aplicar gradient clipping.
        Si es None, no se aplica clipping.
    use_amp : bool
        Si es True y hay GPU, activa mixed precision (AMP) para acelerar el entrenamiento.

    Retorna
    -------
    model : nn.Module
        Modelo con los mejores pesos encontrados (según pérdida de validación).
    """

    # Enviar todo el modelo al dispositivo (GPU/CPU)
    model = model.to(device)

    # Optimizador AdamW (recomendado para arquitecturas tipo Transformer)
    opt = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    # GradScaler para entrenamiento en mixed precision (solo en GPU)
    scaler = torch.cuda.amp.GradScaler(enabled=(use_amp and device == "cuda"))

    # Variables para seguimiento del mejor modelo (early stopping)
    best_val = float("inf")   # mejor pérdida de validación observada
    best_state = None         # state_dict del mejor modelo
    noimp = 0                 # épocas consecutivas sin mejora

    # ==========================================================
    #                      LOOP DE ÉPOCAS
    # ==========================================================
    for epoch in range(1, max_epochs + 1):

        # ----------------------- ENTRENAMIENTO -----------------------
        model.train()
        tr_loss = 0.0

        for xb, yb in dl_tr:
            xb, yb = xb.to(device), yb.to(device)

            # Reset del gradiente
            opt.zero_grad(set_to_none=True)

            # Forward con AMP opcional
            with torch.cuda.amp.autocast(enabled=(use_amp and device == "cuda")):
                # El modelo compacto incluye: encoder → pool → head
                yhat = model(xb).view(-1)  # salida (B,)
                loss = nn.functional.mse_loss(yhat, yb)

            # Backpropagation con GradScaler
            scaler.scale(loss).backward()
            scaler.unscale_(opt)  # necesario antes del clipping

            # Clipping de gradiente para evitar explosiones
            if grad_clip is not None:
                nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            # Paso de optimización
            scaler.step(opt)
            scaler.update()

            # Acumulación de la pérdida ponderada por el tamaño del batch
            tr_loss += loss.item() * xb.size(0)

        # Promedio de pérdida de entrenamiento por muestra
        tr_loss /= len(dl_tr.dataset)

        # ----------------------- VALIDACIÓN -----------------------
        model.eval()
        va_loss = 0.0

        with torch.no_grad():
            for xb, yb in dl_va:
                xb, yb = xb.to(device), yb.to(device)

                yhat = model(xb).view(-1)
                va_loss += nn.functional.mse_loss(yhat, yb).item() * xb.size(0)

        # Promedio de pérdida de validación por muestra
        va_loss /= len(dl_va.dataset)

        # Log de la época
        print(f"Epoch {epoch:03d}  train={tr_loss:.6e}  valid={va_loss:.6e}")

        # ----------------------- EARLY STOPPING -----------------------
        if va_loss < best_val - 1e-9:
            # Mejora en validación: se guarda el mejor modelo hasta ahora
            best_val = va_loss
            noimp = 0
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
        else:
            # No hubo mejora: se incrementa el contador
            noimp += 1
            if noimp >= patience:
                print("Early stopping por falta de mejora en validación.")
                break

    # Restaurar los mejores pesos encontrados
    if best_state is not None:
        model.load_state_dict(best_state)

    return model


### Loop de entrenamiento para tuneo de HP

In [ ]:
import torch.nn as nn

device = "cuda" if torch.cuda.is_available() else "cpu"

# =========================================

# Tamaño de ventana y cantidad de features
T = windows_size   # longitud de la ventana temporal
F = len(features_90)   # número de features por paso
bs = 256 # batch size para entrenamiento

# Pooling compartido (sin parámetros entrenables)
pool = TemporalPooling("mean").to(device)

# Diccionarios para guardar resultados del entrenamiento
models      = {}
scalers_y   = {}
ytr_pred_d  = {}
yva_pred_d  = {}
yte_pred_d  = {}

for fold in k_folds:
    model_key = f"transformer_fold_{fold}"
    #
    # Si ya existe en la tabla de métricas HP, omitimos el entrenamiento
    if ("transformers_metrics_hp" in globals()
        and transformers_metrics_hp is not None
        and model_key in transformers_metrics_hp.index):
        print(f"Omitimos este entrenamiento: {model_key} ya existe en transformers_metrics_hp")
        continue

    print(f"\n=== Entrenando modelo (HP): {model_key} ===")

    # ------------------------------------------------------------
    # 2) Recuperar X e y del fold desde diccionarios
    #    Xtr, Xva, Xte están en 3D: (N, T, F)
    #    y_*_sc[fold] está en 1D: (N,)
    # ------------------------------------------------------------
    Xtr_3d = Xtr[fold]
    Xva_3d = Xva[fold]
    Xte_3d = Xte[fold]

    ytr = y_train_sc[fold]
    yva = y_valid_sc[fold]
    yte = y_test_sc[fold]

    # Aplanar 3D -> 2D para make_loaders: (N, T*F)
    Xtr_flat = Xtr_3d.reshape(Xtr_3d.shape[0], -1)
    Xva_flat = Xva_3d.reshape(Xva_3d.shape[0], -1)
    Xte_flat = Xte_3d.reshape(Xte_3d.shape[0], -1)

    # 3) Scaler de y solo con los datos de entrenamiento del fold
    scaler_y_fold = get_y_scaler(ytr)

    # 4) Crear DataLoaders para entrenamiento y validación del fold
    dl_tr_fold, dl_va_fold = make_loaders(
        Xtr_flat, ytr,
        Xva_flat, yva,
        T=T, F=F,
        y_scaler=scaler_y_fold,
        bs=bs
    )

    # ------------------------------------------------------------
    # 5) Construir el modelo compacto del fold:
    #    encoders[fold] + pool (compartido) + heads[fold]
    # ------------------------------------------------------------
    encoder_fold = encoders[fold]
    head_fold    = heads[fold]

    model_fold = nn.Sequential(
        encoder_fold,  # (B, T, F) → (B, T, D)
        pool,          # (B, T, D) → (B, D)
        head_fold      # (B, D)   → (B,)
    )

    # 6) Entrenar el modelo del fold
    model_fold = train_model_hp(
        model_fold,
        dl_tr_fold,
        dl_va_fold,
        device=device,
        lr=base_hparams["lr"],
        weight_decay=base_hparams["weight_decay"],
        max_epochs=base_hparams["max_epochs"],
        patience=base_hparams["patience"],
        grad_clip=base_hparams["grad_clip"],
        use_amp=base_hparams["use_amp"],
    )

    # ------------------------------------------------------------
    # 7) Predicciones finales en train / valid / test para este fold
    # ------------------------------------------------------------
    ytr_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xtr_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    yva_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xva_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    yte_pred = predict_set(
        encoder_fold,
        pool,
        head_fold,
        Xte_flat,
        T, F,
        device,
        batch_size=4096,
        y_scaler=scaler_y_fold
    )

    # ------------------------------------------------------------
    # 8) Cálculo de métricas por fold
    # ------------------------------------------------------------
    metrics_tr = evaluate_model(
        model=None,
        X=None,
        y_true=ytr,
        y_pred=ytr_pred
    )

    metrics_va = evaluate_model(
        model=None,
        X=None,
        y_true=yva,
        y_pred=yva_pred
    )

    metrics_te = evaluate_model(
        model=None,
        X=None,
        y_true=yte,
        y_pred=yte_pred
    )

    # ------------------------------------------------------------
    # 9) Guardar métricas en el DataFrame de tuning HP
    # ------------------------------------------------------------
    if "transformers_metrics_hp" in globals() and transformers_metrics_hp is not None:
        for split, m in [("train", metrics_tr), ("valid", metrics_va), ("test", metrics_te)]:
            for k, v in m.items():
                transformers_metrics_hp.loc[model_key, f"{split}_{k}"] = v
    # ------------------------------------------------------------
    # 10) Guardar el modelo y outputs en diccionarios (RAM)
    # ------------------------------------------------------------
    models[fold]     = model_fold
    scalers_y[fold]  = scaler_y_fold
    ytr_pred_d[fold] = ytr_pred
    yva_pred_d[fold] = yva_pred
    yte_pred_d[fold] = yte_pred

    # ------------------------------------------------------------
    # 11) Guardar checkpoint en DISCO usando rutas_modelos[fold]
    # ------------------------------------------------------------
    ruta_ckpt = rutas_modelos_hp[fold]["model_path"]

    checkpoint = {
        "model_state":   model_fold.state_dict(),
        "encoder_state": encoder_fold.state_dict(),
        "head_state":    head_fold.state_dict(),
        "scaler_y":      scaler_y_fold,
        "metrics_train": metrics_tr,
        "metrics_valid": metrics_va,
        "metrics_test":  metrics_te,
        "hparams": {
            "T": T,
            "F": F,
            "lr": base_hparams["lr"],
            "weight_decay": base_hparams["weight_decay"],
            "max_epochs": base_hparams["max_epochs"],
            "patience": base_hparams["patience"],
            "grad_clip": base_hparams["grad_clip"],
            "use_amp": base_hparams["use_amp"],
            "pooling": "mean",
            "batch_size": bs,
        },
        # "optimizer_state": optimizer.state_dict(),  # si lo querés guardar
    }

    torch.save(checkpoint, ruta_ckpt)
    print(f"✔ Checkpoint guardado para fold {fold} en: {ruta_ckpt}")
#18min


=== Entrenando modelo: transformer_fold_1 ===
Epoch 001  train=2.385897e-01  valid=6.182196e-01
Epoch 002  train=1.960405e-01  valid=6.269121e-01
Epoch 003  train=1.676874e-01  valid=6.262041e-01
Epoch 004  train=1.463421e-01  valid=6.316790e-01
Epoch 005  train=1.283954e-01  valid=6.712022e-01
Epoch 006  train=1.153954e-01  valid=6.587794e-01
Epoch 007  train=1.048695e-01  valid=6.964915e-01
Epoch 008  train=9.340970e-02  valid=6.868323e-01
Epoch 009  train=8.742683e-02  valid=6.808304e-01
Early stopping por falta de mejora en validación.
✔ Checkpoint guardado para fold 1 en: /content/drive/MyDrive/neural_profit/5_transformer_90_model/5_3_model_transformer/5_3_0_trained_k_models/transformer_fold_1.pt

=== Entrenando modelo: transformer_fold_2 ===
Epoch 001  train=2.372889e-01  valid=2.653519e-01
Epoch 002  train=1.976300e-01  valid=2.639960e-01
Epoch 003  train=1.686139e-01  valid=2.961413e-01
Epoch 004  train=1.471843e-01  valid=2.900800e-01
Epoch 005  train=1.301977e-01  valid=3.05

### 10.2. Estrategia para tuning

Para no hacer algo imposible de correr y romper el colab:

1. Tuneamos solo sobre un solo fold (por ejemplo fold_1), usando solo train + valid.

2. Dejamos el resto de folds (2–5) como evaluación “honesta” del mejor set de hiperparámetros.

3. Limitamos el coste:

- `max_epochs` bajo (5–10) durante el tuning.
- Menos `n_trials` (por ejemplo 20–30 pruebas)

### 10.3. Hiperparámetros claves

Los nombres pueden variar según tu implementación, pero típicamente:

1. `d_model` (embedding / hidden size)

    - Qué controla: dimensión de los vectores internos.
    - Valores razonables: 32, 64, 128, 256.
    - Más grande = más capacidad, pero más riesgo de overfitting y más consumo.

2. `nhead` (número de cabezas de atención)

    - Qué controla: cuántas “sub-atenciones” paralelas tiene cada bloque.
    - Debe dividir exactamente a d_model.
    - Valores típicos: 2, 4, 8.
    - Para d_model=64 → 2 u 4; para d_model=128 → 4 u 8, etc.

3. `num_encoder_layers` / `num_decoder_layers`

    - Qué controla: profundidad del modelo.
    - Valores razonables: 2, 3, 4.
    - Más capas = más capacidad, pero entrenar peor y más riesgo de sobreajuste.

4. `dim_feedforward` (tamaño del MLP interno de cada bloque)

    - Normalmente 2–4× d_model.
    - Ej.: para d_model=64, probar 128, 256, 512.

5. `dropout`

    - Tu principal herramienta de regularización.
    - Rango sano para series financieras: 0.1–0.4.

6. `learning_rate`

    - Clave: muchas veces mejora más que cambiar la arquitectura.
    - Rango típico: 1e-4 a 3e-3 (log-uniforme).

7. `batch_size`

    - Influye en estabilidad del entrenamiento.
    - Valores razonables: 64, 128, 256 (según tu VRAM).

### 10.4. Espacio de búsqueda (ligera)

Para que sea corrible en Colab, algo como:

  `d_model`: [64, 128]

  `nhead`: [2, 4] (compatibles con d_model)

  `num_layers`: [2, 3] (si usás encoder-only, un solo parámetro)

  `dim_feedforward`: [128, 256, 512]

  `dropout`: 0.1–0.4

  `learning_rate`: 1e-4–2e-3 (log)

  `batch_size`: [64, 128]

Con eso ya tenés suficiente variación para ver si baja RMSE / sube R² sin matar Colab.

In [ ]:
encoders

In [ ]:
def build_encoders_for_folds(k_folds, hparams, device):
    encoders = {}
    for k in k_folds:
        encoders[k] = TimeSeriesEncoder(
            input_dim=n_features_90,
            d_model=hparams["d_model"],
            nhead=hparams["nhead"],
            num_layers=hparams["num_layers"],
            dim_feedforward=hparams["dim_feedforward"],
            dropout=hparams["dropout"],
            activation=hparams["activation"],
        ).to(device)
    return encoders

In [ ]:
import optuna

def objective(trial):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    # 1) Espacio de hiperparámetros
    hparams = {
        "input_dim":       n_features_90,
        "d_model":         trial.suggest_categorical("d_model", [64, 128, 256]),
        "nhead":           trial.suggest_categorical("nhead", [2, 4, 8]),
        "num_layers":      trial.suggest_int("num_layers", 1, 3),
        "dim_feedforward": trial.suggest_categorical("dim_feedforward", [128, 256, 512]),
        "dropout":         trial.suggest_float("dropout", 0.05, 0.3),
        "activation":      "gelu",
    }

    # 2) Construir encoder SOLO para el fold 1 (para tuning)
    encoder_1 = TimeSeriesEncoder(
        input_dim=hparams["input_dim"],
        d_model=hparams["d_model"],
        nhead=hparams["nhead"],
        num_layers=hparams["num_layers"],
        dim_feedforward=hparams["dim_feedforward"],
        dropout=hparams["dropout"],
        activation=hparams["activation"],
    ).to(device)

    # 3) Pool + head del fold 1
    pool = GlobalAveragePool()
    head_1 = nn.Sequential(
        nn.Linear(hparams["d_model"], hparams["d_model"] // 2),
        nn.ReLU(),
        nn.Linear(hparams["d_model"] // 2, 1)
    ).to(device)

    model_1 = nn.Sequential(encoder_1, pool, head_1)

    # 4) DataLoaders de entrenamiento/validación del fold 1
    dl_tr = globals()["dl_train_1"]
    dl_va = globals()["dl_valid_1"]

    # 5) Entrenar con menos épocas durante tuning
    model_1 = train_model(
        model_1,
        dl_tr,
        dl_va,
        device=device,
        lr=trial.suggest_float("lr", 1e-4, 3e-3, log=True),
        weight_decay=trial.suggest_float("weight_decay", 1e-6, 1e-3, log=True),
        max_epochs=15,
        patience=4,
        grad_clip=1.0,
        use_amp=True,
    )

    # 6) Calcular pérdida de validación como métrica a minimizar
    model_1.eval()
    va_loss = 0.0
    with torch.no_grad():
        for xb, yb in dl_va:
            xb, yb = xb.to(device), yb.to(device)
            yhat = model_1(xb).view(-1)
            loss = nn.functional.mse_loss(yhat, yb)
            va_loss += loss.item() * xb.size(0)
    va_loss /= len(dl_va.dataset)

    return va_loss

ModuleNotFoundError: No module named 'optuna'

In [ ]:
#Paso 3 – Lanzar el estudio de tuning
study = optuna.create_study(direction="minimize")
study.optimize(objective, n_trials=20)

best_params = study.best_params
best_params


In [ ]:
#Paso 4 – Con los mejores hiperparámetros, reentrenás “en serio”
##Usás los best_params para instanciar tus encoder_k (como en el pipeline normal, pero con esos valores).
##Repetís el Pipeline 1 para todos los folds con esos hiperparámetros fijos “óptimos”.